In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.figsize': (14, 6)})

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

# ---- Configure dataset here ----
DATASET = "celeba"  # or "celebahq"
base = repo_root / "output" / "smile_classification" / DATASET
certify_dir = base / "certify"

# Mode tags match CertifyPaths.from_config:
#   {pixel|latent}_{manifold|isotropic}
MODES = ["pixel_isotropic", "pixel_manifold", "latent_isotropic", "latent_manifold"]
SIGMAS = ["sigma_0_25", "sigma_0_50", "sigma_0_75", "sigma_1_00"]

def sigma_val(s):
    return float(s.replace("sigma_", "").replace("_", "."))

def load_metrics(folder):
    """Load metrics.json from a certify experiment folder."""
    if not folder.exists():
        return None
    p = folder / "metrics.json"
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None

def load_results_csv(folder):
    """Load per-sample results.csv."""
    p = folder / "results.csv"
    if p.exists():
        return pd.read_csv(p)
    return None

def mode_label(mode):
    """Pretty label: 'pixel_manifold' -> 'Pixel Manifold'."""
    return mode.replace('_', ' ').title()

# Discover what results exist
found = []
for mode in MODES:
    for sig in SIGMAS:
        p = certify_dir / mode / sig
        m = load_metrics(p)
        if m:
            found.append((mode, sig, sigma_val(sig)))

print(f"Dataset: {DATASET}")
print(f"Base: {base}")
print(f"Found {len(found)} experiment results:")
for mode, sig, sv in found:
    print(f"  {mode_label(mode):25s}  σ={sv}")

---
## 1. Grand Summary — All Experiments

In [ ]:
rows = []
for mode in MODES:
    for sig in SIGMAS:
        m = load_metrics(certify_dir / mode / sig)
        if m is None:
            continue
        rows.append({
            'mode': mode_label(mode),
            'space': mode.split('_')[0].title(),
            'smoothing': mode.split('_')[1].title(),
            'sigma': sigma_val(sig),
            'certified_acc': m.get('certified_accuracy', 0),
            'abstain_rate': m.get('abstain_rate', 0),
            'mean_radius': m.get('mean_radius', 0),
            'median_radius': m.get('median_radius', 0),
            'max_radius': m.get('max_radius', 0),
            'std_radius': m.get('std_radius', 0),
            'total': m.get('total_test_samples', 0),
            'certified_correct': m.get('certified_correct', 0),
            'smile_acc': m.get('class_smile_accuracy', 0),
            'no_smile_acc': m.get('class_no_smile_accuracy', 0),
            'smile_radius': m.get('class_smile_mean_radius', 0),
            'no_smile_radius': m.get('class_no_smile_mean_radius', 0),
        })

if rows:
    df = pd.DataFrame(rows)
    print("=" * 120)
    print(f"GRAND SUMMARY — {DATASET.upper()} SMILE CERTIFICATION")
    print("=" * 120)
    display(df[['mode', 'sigma', 'certified_acc', 'mean_radius', 'abstain_rate',
                'certified_correct', 'total', 'smile_acc', 'no_smile_acc']].round(4))
else:
    print("No results found. Copy server results to:", certify_dir)

---
## 2. Isotropic vs Manifold — Pixel Space

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    pixel = df[df['space'] == 'Pixel'].copy()
    
    if len(pixel) > 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, metric, title in zip(axes,
            ['certified_acc', 'mean_radius', 'abstain_rate'],
            ['Certified Accuracy', 'Mean Certified Radius', 'Abstention Rate']):
            for sm in ['Isotropic', 'Manifold']:
                sub = pixel[pixel['smoothing'] == sm].sort_values('sigma')
                if len(sub) > 0:
                    ax.plot(sub['sigma'], sub[metric], 'o-', label=sm, linewidth=2, markersize=8)
            ax.set_xlabel('σ'); ax.set_ylabel(title); ax.set_title(title)
            ax.legend(); ax.grid(True, alpha=0.3)
        plt.suptitle(f'{DATASET.upper()} — Pixel Space: Isotropic vs Manifold', fontsize=14, y=1.02)
        plt.tight_layout(); plt.show()
    else:
        print("No pixel-space results found.")

---
## 3. Isotropic vs Manifold — Latent Space

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    latent = df[df['space'] == 'Latent'].copy()
    
    if len(latent) > 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, metric, title in zip(axes,
            ['certified_acc', 'mean_radius', 'abstain_rate'],
            ['Certified Accuracy', 'Mean Certified Radius', 'Abstention Rate']):
            for sm in ['Isotropic', 'Manifold']:
                sub = latent[latent['smoothing'] == sm].sort_values('sigma')
                if len(sub) > 0:
                    ax.plot(sub['sigma'], sub[metric], 'o-', label=sm, linewidth=2, markersize=8)
            ax.set_xlabel('σ'); ax.set_ylabel(title); ax.set_title(title)
            ax.legend(); ax.grid(True, alpha=0.3)
        plt.suptitle(f'{DATASET.upper()} — Latent Space: Isotropic vs Manifold', fontsize=14, y=1.02)
        plt.tight_layout(); plt.show()
    else:
        print("No latent-space results found.")

---
## 4. Pixel vs Latent — All 4 Methods Compared

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = {'Pixel Isotropic': '#e41a1c', 'Pixel Manifold': '#377eb8',
              'Latent Isotropic': '#ff7f00', 'Latent Manifold': '#4daf4a'}
    
    for ax, metric, title in zip(axes,
        ['certified_acc', 'mean_radius', 'abstain_rate'],
        ['Certified Accuracy', 'Mean Certified Radius', 'Abstention Rate']):
        for mode_name in colors:
            sub = df[df['mode'] == mode_name].sort_values('sigma')
            if len(sub) > 0:
                ax.plot(sub['sigma'], sub[metric], 'o-', label=mode_name,
                        color=colors[mode_name], linewidth=2, markersize=8)
        ax.set_xlabel('σ'); ax.set_ylabel(title); ax.set_title(title)
        ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} — All 4 Methods Compared', fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

---
## 5. Per-Class (Smile / No-Smile) Comparison

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = {'Pixel Isotropic': '#e41a1c', 'Pixel Manifold': '#377eb8',
              'Latent Isotropic': '#ff7f00', 'Latent Manifold': '#4daf4a'}
    
    for ax, cls_metric, cls_name in zip(axes,
        ['smile_acc', 'no_smile_acc'], ['Smile', 'No-Smile']):
        for mode_name in colors:
            sub = df[df['mode'] == mode_name].sort_values('sigma')
            if len(sub) > 0:
                ax.plot(sub['sigma'], sub[cls_metric], 'o-', label=mode_name,
                        color=colors[mode_name], linewidth=2, markersize=8)
        ax.set_xlabel('σ'); ax.set_ylabel('Certified Accuracy')
        ax.set_title(f'{cls_name} Class — Certified Accuracy')
        ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} — Per-Class Certified Accuracy', fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

---
## 6. Certification Radius Distribution (Per-Sample)

In [ ]:
# Load per-sample results for radius histograms at a chosen sigma
TARGET_SIGMA = "sigma_0_50"

csv_data = {}
for mode in MODES:
    df_csv = load_results_csv(certify_dir / mode / TARGET_SIGMA)
    if df_csv is not None:
        csv_data[mode_label(mode)] = df_csv

if csv_data:
    fig, axes = plt.subplots(1, len(csv_data), figsize=(5 * len(csv_data), 5), squeeze=False)
    axes = axes.flat
    
    for ax, (name, df_csv) in zip(axes, csv_data.items()):
        non_abstain = df_csv[df_csv['abstained'] == False]
        radii = non_abstain['radius'].values
        ax.hist(radii, bins=30, alpha=0.7, edgecolor='k')
        ax.axvline(radii.mean(), color='red', linestyle='--', label=f'mean={radii.mean():.3f}')
        ax.set_xlabel('Certified Radius'); ax.set_ylabel('Count')
        ax.set_title(name); ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} — Radius Distribution at σ={sigma_val(TARGET_SIGMA)}',
                 fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print(f"No per-sample results.csv found at σ={sigma_val(TARGET_SIGMA)}")

---
## 7. Certified Accuracy vs Radius Trade-off

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    colors = {'Pixel Isotropic': '#e41a1c', 'Pixel Manifold': '#377eb8',
              'Latent Isotropic': '#ff7f00', 'Latent Manifold': '#4daf4a'}
    
    fig, ax = plt.subplots(figsize=(10, 7))
    for mode_name in colors:
        sub = df[df['mode'] == mode_name].sort_values('sigma')
        if len(sub) > 0:
            ax.plot(sub['mean_radius'], sub['certified_acc'], 'o-', label=mode_name,
                    color=colors[mode_name], linewidth=2, markersize=10)
            # Annotate with sigma values
            for _, r in sub.iterrows():
                ax.annotate(f'σ={r["sigma"]}', (r['mean_radius'], r['certified_acc']),
                            textcoords='offset points', xytext=(6, 6), fontsize=8)
    
    ax.set_xlabel('Mean Certified Radius', fontsize=13)
    ax.set_ylabel('Certified Accuracy', fontsize=13)
    ax.set_title(f'{DATASET.upper()} — Accuracy vs Radius Trade-off', fontsize=14)
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
## 8. Heatmap — Certified Accuracy by Mode & σ

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(index='mode', columns='sigma', values='certified_acc', aggfunc='first')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'σ={s}' for s in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.1%}', ha='center', va='center', fontsize=11,
                        fontweight='bold')
    
    plt.colorbar(im, label='Certified Accuracy')
    ax.set_title(f'{DATASET.upper()} — Certified Accuracy Heatmap', fontsize=14)
    plt.tight_layout(); plt.show()

---
## 9. Per-Sample Correct/Incorrect Breakdown

In [ ]:
# Stacked bar: correct / incorrect / abstained per mode at a chosen sigma
TARGET_SIGMA = "sigma_0_50"

bar_rows = []
for mode in MODES:
    m = load_metrics(certify_dir / mode / TARGET_SIGMA)
    if m:
        total = m['total_test_samples']
        correct = m['certified_correct']
        abstained = m['abstained_samples']
        wrong = total - correct - abstained
        bar_rows.append({
            'mode': mode_label(mode),
            'Correct': correct / total,
            'Wrong': wrong / total,
            'Abstained': abstained / total,
        })

if bar_rows:
    df_bar = pd.DataFrame(bar_rows).set_index('mode')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    df_bar.plot(kind='bar', stacked=True, ax=ax,
                color=['#4daf4a', '#e41a1c', '#999999'], alpha=0.8)
    ax.set_ylabel('Fraction'); ax.set_ylim(0, 1.05)
    ax.set_title(f'{DATASET.upper()} — Outcome Breakdown at σ={sigma_val(TARGET_SIGMA)}', fontsize=14)
    ax.legend(loc='upper right'); ax.grid(True, alpha=0.2, axis='y')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()
else:
    print(f"No results at σ={sigma_val(TARGET_SIGMA)}")

---
## 10. Per-Class Radius Comparison

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = {'Pixel Isotropic': '#e41a1c', 'Pixel Manifold': '#377eb8',
              'Latent Isotropic': '#ff7f00', 'Latent Manifold': '#4daf4a'}
    
    for ax, cls_metric, cls_name in zip(axes,
        ['smile_radius', 'no_smile_radius'], ['Smile', 'No-Smile']):
        for mode_name in colors:
            sub = df[df['mode'] == mode_name].sort_values('sigma')
            if len(sub) > 0:
                ax.plot(sub['sigma'], sub[cls_metric], 'o-', label=mode_name,
                        color=colors[mode_name], linewidth=2, markersize=8)
        ax.set_xlabel('σ'); ax.set_ylabel('Mean Certified Radius')
        ax.set_title(f'{cls_name} Class — Mean Radius')
        ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} — Per-Class Certified Radius', fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()